## Librerías

In [1]:
import pandas as pd
from pathlib import Path
import sys
import warnings

In [2]:
sys.path.append("../src")
from entities import EntityResolver
from graphs import HeterogeneousGraphBuilder
from graphs import GraphFeatureExtractor

In [3]:
# Ignoramos warnings de spaCy para mantener la salida limpia
warnings.filterwarnings("ignore")

## PATHS

In [4]:
BASE_DIR = Path.cwd().parent
CSV_CHATS = BASE_DIR / "data" / "processed" / "chats_completos.csv"
EDGES_CSV_RAW = BASE_DIR / "data" / "processed" / "grafo_aristas.csv"
EDGES_CSV_CLEAN = BASE_DIR / "data" / "processed" / "grafo_aristas_resuelto.csv"
NODES_FEATURES_CSV = BASE_DIR / "data" / "processed" / "nodos_features.csv"

## Orquestación

In [5]:
if not EDGES_CSV_CLEAN.exists():
    print("Iniciando Pipeline de Grafo Heterogéneo...")
    df_chats = pd.read_csv(CSV_CHATS)
    
    # Fase A: Construcción de la Topología Base
    constructor = HeterogeneousGraphBuilder()
    df_grafo_crudo = constructor.build_raw_graph(df_chats)
    df_grafo_crudo.to_csv(EDGES_CSV_RAW, index=False)
    
    # Fase B: Resolución de Entidades (Entity Resolution)
    resolver = EntityResolver()
    df_grafo_limpio = resolver.consolidar_grafo(
        df_grafo=df_grafo_crudo, 
        columnas_entidades=["Origen", "Destino"]
    )
    df_grafo_limpio.to_csv(EDGES_CSV_CLEAN, index=False)
    print("Pipeline completado. Grafo consolidado y guardado.")

else:
    print("Cargando grafo limpio desde caché...")
    df_grafo_limpio = pd.read_csv(EDGES_CSV_CLEAN)

Cargando grafo limpio desde caché...


In [6]:
# 1. Cargamos el grafo crudo que sacamos en el paso anterior
df_grafo_crudo = pd.read_csv(EDGES_CSV_RAW)

# 2. Instanciamos nuestra clase (podemos inyectarle nuevos alias si descubrimos más)
resolver = EntityResolver()

# 3. Consolidamos la red matemática
df_grafo_limpio = resolver.consolidar_grafo(
    df_grafo=df_grafo_crudo, 
    columnas_entidades=["Origen", "Destino"]
)

### Auditoria

In [7]:
print(f"\nTotal de aristas únicas en la red: {len(df_grafo_limpio)}")

print("\nTop 5 - Relaciones Físicas (COMUNICA_CON):")
display(df_grafo_limpio[df_grafo_limpio['Tipo_Relacion'] == 'COMUNICA_CON'].head(5))

print("\nTop 5 - Relaciones de Influencia (MENCIONA_A):")
display(df_grafo_limpio[df_grafo_limpio['Tipo_Relacion'] == 'MENCIONA_A'].head(5))


Total de aristas únicas en la red: 519

Top 5 - Relaciones Físicas (COMUNICA_CON):


,Origen,Destino,Tipo_Relacion,Peso
0,MIGUEL PALOMERO,RODOLFO REYES,COMUNICA_CON,94
1,RODOLFO REYES,JULIO MARTINEZ SOLA,COMUNICA_CON,86
2,ROBERTO ROSELLI,RODOLFO REYES,COMUNICA_CON,77
3,JULIO MARTINEZ SOLA,RODOLFO REYES,COMUNICA_CON,76
4,RODOLFO REYES,MIGUEL PALOMERO,COMUNICA_CON,68



Top 5 - Relaciones de Influencia (MENCIONA_A):


,Origen,Destino,Tipo_Relacion,Peso
6,RODOLFO REYES,JULIO MARTINEZ SOLA,MENCIONA_A,30
7,JULIO MARTINEZ SOLA,RODOLFO REYES,MENCIONA_A,29
8,ROBERTO ROSELLI,JULIO MARTINEZ MARTINEZ,MENCIONA_A,23
9,ROBERTO ROSELLI,JULIO MARTINEZ SOLA,MENCIONA_A,22
10,ROBERTO ROSELLI,RODOLFO REYES,MENCIONA_A,22


## Creación y cálculo de nodos

In [8]:
# Calculamos las métricas topológicas si no lo hemos hecho antes
if not NODES_FEATURES_CSV.exists():
    extractor = GraphFeatureExtractor()
    df_features = extractor.extract_features(df_grafo_limpio)
    
    # Guardamos nuestro dataset tabular final para Machine Learning
    df_features.to_csv(NODES_FEATURES_CSV, index=False)
    print("Métricas de red calculadas y guardadas con éxito.")
else:
    print("Cargando features de red desde caché...")
    df_features = pd.read_csv(NODES_FEATURES_CSV)

Calculando métricas topológicas (Feature Engineering)...
Métricas de red calculadas y guardadas con éxito.


In [9]:
# Visualizamos a los pesos pesados de la red según el algoritmo PageRank
print("\nTop 10 Nodos más importantes de la red (según PageRank):")
display(df_features.head(10))


Top 10 Nodos más importantes de la red (según PageRank):


,Nodo,In_Degree,Out_Degree,Betweenness,PageRank,Hub_Score,Authority_Score
0,RODOLFO REYES,0.123506,0.378486,0.180645,0.081142,0.185744,0.241222
1,JULIO MARTINEZ SOLA,0.083665,0.314741,0.089418,0.048253,0.199699,0.123368
2,ROBERTO ROSELLI,0.075697,0.354582,0.099187,0.038951,0.222246,0.067471
3,MIGUEL PALOMERO,0.039841,0.107570,0.029355,0.023670,0.181608,0.060812
4,JULIO MARTINEZ MARTINEZ,0.067729,0.075697,0.020382,0.022128,0.018402,0.060466
5,ZAPATERO,0.023904,0.000000,0.000000,0.008930,-0.000000,0.033638
6,RODOLFO REYES ROJAS,0.027888,0.000000,0.000000,0.008822,-0.000000,0.030470
7,JULIO,0.035857,0.000000,0.000000,0.008497,-0.000000,0.022198
8,ROBERTO ROSELLI MIELE,0.031873,0.000000,0.000000,0.007866,-0.000000,0.011736
9,FELIPE BACA,0.011952,0.031873,0.015283,0.007194,0.017373,0.014063


In [10]:
# Visualizamos a los principales Brokers (Highest Betweenness)
print("\nTop 5 Intermediarios/Brokers (según Betweenness Centrality):")
display(df_features.sort_values(by="Betweenness", ascending=False).head(5))


Top 5 Intermediarios/Brokers (según Betweenness Centrality):


,Nodo,In_Degree,Out_Degree,Betweenness,PageRank,Hub_Score,Authority_Score
0,RODOLFO REYES,0.123506,0.378486,0.180645,0.081142,0.185744,0.241222
2,ROBERTO ROSELLI,0.075697,0.354582,0.099187,0.038951,0.222246,0.067471
1,JULIO MARTINEZ SOLA,0.083665,0.314741,0.089418,0.048253,0.199699,0.123368
3,MIGUEL PALOMERO,0.039841,0.107570,0.029355,0.023670,0.181608,0.060812
4,JULIO MARTINEZ MARTINEZ,0.067729,0.075697,0.020382,0.022128,0.018402,0.060466
